# FAISS


**FAISS** (*Facebook AI Similarity Search*) is an **open-source library from Meta AI** for finding the vectors most similar to a query vector. It is written in C++ with Python bindings, and can also run on a GPU. It is popular for RAG because:

- **It is very fast:** it is built only for nearest-neighbour search, and is heavily optimised for it.
- **It scales:** with the right index type it can search millions, or even billions, of vectors.
- **You choose the trade-off:** exact or approximate search, more speed or less memory, through many **index types**.
- **No server:** `pip install faiss-cpu` and it runs inside your Python process.
- **LangChain support:** the `FAISS` vector store adds what the library is missing (the chunk text, metadata, and saving to disk), so it works like any other LangChain vector store.

The key difference from Chroma: FAISS on its own is a **vector index library**, not a database. It stores **only vectors**, each one at a numbered position (0, 1, 2, ...). It has no text, no metadata, no server and no automatic saving.

> **In one sentence:** FAISS is a very fast library for finding the nearest vectors, and LangChain wraps it so it works like a normal vector store.

This notebook uses `faiss-cpu` 1.15.x with LangChain's `FAISS` class from `langchain_community`.

## FAISS vs. Chroma

| | FAISS | Chroma |
|---|---|---|
| **What it is** | A vector search **library** | A vector **database** |
| **What it stores** | Vectors only. LangChain keeps the text and metadata next to it | Vectors, text and metadata together |
| **Saving to disk** | Manual: `save_local()` / `load_local()` | Automatic with `persist_directory` |
| **Index types** | Many: Flat (exact), IVF, HNSW, PQ, ... | HNSW |
| **Metadata filtering** | Done by LangChain **after** the search | Built in (`where`, `where_document`) |
| **Update a chunk** | Delete it, then add it again (no upsert) | `upsert()` / `update()` |
| **Server / several users** | No: one Python process | Yes: `HttpClient` or Chroma Cloud |
| **GPU** | Yes (`faiss-gpu`) | No |
| **Best for** | Fast local search, large collections that rarely change, trying out index types | Apps that need CRUD, filtering and persistence out of the box |

Both plug into LangChain the same way (`from_documents`, `similarity_search`, `as_retriever`), so switching between them only changes a few lines of code.

## How LangChain wraps FAISS

The FAISS index only knows **positions and vectors**. LangChain adds two small pieces so it can return `Document` objects:

```
LangChain FAISS vector store
 │
 ├── index                  faiss.IndexFlatL2: one vector per chunk, at positions 0, 1, 2, ...
 │
 ├── index_to_docstore_id   {0: "chunk_0", 1: "chunk_1", 2: "chunk_2", ...}
 │
 └── docstore               InMemoryDocstore: {"chunk_0": Document(page_content, metadata), ...}
```

What happens in `similarity_search(query, k=3)`:

```
"What is reinforcement learning?"
        │
        ▼
 1. Embed the question          embeddings.embed_query(query)
        │
        ▼
 2. FAISS searches the index    → the 3 closest positions, e.g. [7, 2, 11], and their distances
        │
        ▼
 3. Positions → ids             index_to_docstore_id   → ["chunk_7", "chunk_2", "chunk_11"]
        │
        ▼
 4. Ids → Documents             docstore               → text + metadata of each chunk
```

`save_local(FAISS_PATH)` writes two files to `vectorstore/faiss_index/`:

```
vectorstore/faiss_index/
├── index.faiss     ← the FAISS index (the vectors), written by faiss.write_index
└── index.pkl       ← docstore + index_to_docstore_id, saved with Python's pickle
```

## Index types

FAISS offers many index types. They trade off **speed**, **memory** and **accuracy**:

| Index | How it searches | Exact? | Needs training? | Use it when |
|---|---|---|---|---|
| `IndexFlatL2` / `IndexFlatIP` | Compares the question with **every** vector (brute force) | Yes | No | Small to medium collections, e.g. up to a few hundred thousand chunks. **LangChain's default** |
| `IndexIVFFlat` | Groups the vectors into `nlist` clusters, then searches only the `nprobe` closest clusters | No (approximate) | Yes | Millions of vectors, where brute force gets slow |
| `IndexHNSWFlat` | Walks a graph of neighbouring vectors (the same idea as Chroma's index) | No (approximate) | No | Fast, accurate search when you have memory to spare |
| `IndexIVFPQ` | IVF plus **product quantization**: each vector is compressed to a few bytes | No (approximate) | Yes | Hundreds of millions of vectors, or limited RAM |

How IVF speeds up search:

```
IndexIVFFlat with nlist = 4 clusters and nprobe = 1

   cluster A: ● ● ●          cluster B: ● ●
   cluster C: ● ● ● ●        cluster D: ● ● ●   ← the question is closest to D's centre,
                                                   so only D's vectors are compared
```

A larger `nprobe` searches more clusters: more accurate, but slower.

**Training:** IVF and PQ indexes must learn their clusters from sample vectors with `index.train(vectors)` **before** you add anything. Flat and HNSW indexes don't need training.

```python
import faiss

index = faiss.index_factory(1536, "IVF100,Flat")   # 1536 dims, 100 clusters
index.train(vectors)                               # learn the clusters first
index.add(vectors)
index.nprobe = 10                                  # search the 10 closest clusters
```

**Rule of thumb:** start with the default flat index. It is exact and simple, and it is fast enough for most course projects and prototypes. Move to IVF or HNSW only when search becomes slow.

## Distance metrics

| Index | Measure | Score from `similarity_search_with_score` |
|---|---|---|
| `IndexFlatL2` (the default) | **Squared** Euclidean distance | A **distance**: **lower = better**, 0 = identical |
| `IndexFlatIP` | Inner product | A **similarity**: **higher = better** |

**Cosine similarity** = inner product of **normalised** vectors. To get cosine scores, ask LangChain to normalise the vectors and use an inner-product index:

```python
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
    normalize_L2=True,                                  # make every vector length 1
    distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT,
)
```

The score is then the cosine similarity: `1.0` = same direction, `0` = unrelated.

OpenAI embeddings are **already normalised**, so the default L2 index and cosine put the results in the **same order**. Only the numbers differ, so the default works fine.

**Relevance scores: two catches**

- **Default L2 index:** `similarity_search_with_relevance_scores` uses `1 - distance / √2`, the same formula as with Chroma. Weak matches with a squared distance above about **1.41** get **negative** scores.
- **Inner-product index:** `similarity_search_with_relevance_scores` treats the similarity as if it were a distance, so the **best** match gets a relevance of about **0**. With an inner-product index, use `similarity_search_with_score` and read the cosine similarity directly.

## Working with a FAISS vector store in LangChain

| Method | What it does |
|---|---|
| `FAISS.from_documents(docs, embedding, ids=...)` | Embed the chunks and build a new store |
| `vectorstore.similarity_search(query, k=4)` | Return the `k` most similar `Document`s (default `k=4`) |
| `vectorstore.similarity_search_with_score(query, k=4)` | Return `(Document, score)` pairs |
| `vectorstore.max_marginal_relevance_search(query, k, fetch_k)` | Return relevant **and** varied chunks (MMR) |
| `vectorstore.add_documents(docs, ids=...)` | Add more chunks to the store |
| `vectorstore.delete(ids=[...])` | Remove chunks by id |
| `vectorstore.merge_from(other_store)` | Add every chunk of another FAISS store to this one |
| `vectorstore.save_local(folder)` | Save `index.faiss` + `index.pkl` to a folder |
| `FAISS.load_local(folder, embeddings, allow_dangerous_deserialization=True)` | Load a saved store |
| `vectorstore.as_retriever(search_kwargs={"k": 3})` | Use it in a RAG chain, like the Chroma retriever |

### Build, save and load

```python
from langchain_community.vectorstores import FAISS
from src.config import FAISS_PATH   # vectorstore/faiss_index

# Build the store from the chunks, then save it to disk
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
)
vectorstore.save_local(FAISS_PATH)

# Later, or in another notebook: load it again without re-embedding
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings,                              # the same embedding model used to build it
    allow_dangerous_deserialization=True,    # index.pkl is a pickle file: only load files you created
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
```

### Things to watch out for

- **Saving is manual.** `add_documents` and `delete` only change the store **in memory**. Call `save_local()` again to keep the changes. Chroma saves automatically, FAISS does not.
- **`allow_dangerous_deserialization=True`.** `index.pkl` is a Python pickle, and loading a pickle can run code. LangChain makes you opt in, so only load folders that you created yourself.
- **No upsert.** Adding an id that already exists raises `ValueError: Tried to add ids that already exist`. By then the vector has **already been added** to the FAISS index, so the store is left inconsistent, and later searches can fail with a `KeyError`. Check the existing ids before adding, and to change a chunk, delete it first:

```python
existing_ids = set(vectorstore.index_to_docstore_id.values())
to_add = [(chunk, id_) for chunk, id_ in zip(new_chunks, new_ids) if id_ not in existing_ids]

if to_add:
    vectorstore.add_documents([chunk for chunk, _ in to_add], ids=[id_ for _, id_ in to_add])
    vectorstore.save_local(FAISS_PATH)
```

- **Same embedding model.** As with Chroma, always load and query with the model that built the index. A different model, or a different vector size, gives an error or meaningless results.

## Metadata filtering

FAISS itself knows nothing about metadata, so **LangChain filters after the search**:

```
vectorstore.similarity_search(query, k=3, filter={"source": str(RAW_TEXT_DIR / "ai_agents.txt")}, fetch_k=20)

 FAISS: the 20 nearest chunks  ─►  LangChain keeps the ones whose metadata matches  ─►  the first 3
```

```python
results = vectorstore.similarity_search(
    "What are the core components of an AI agent?",
    k=3,
    filter={"source": str(RAW_TEXT_DIR / "ai_agents.txt")},
    fetch_k=20,
)
```

The loaders store the full file path in `source`, so build the value from the same `src.config` constant the files were loaded from.

The filter dict supports the operators `$eq`, `$neq`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`, and `$and` / `$or` / `$not` to combine them, for example `{"topic": {"$in": ["ml", "dl"]}}`. You can also pass a function: `filter=lambda metadata: metadata["source"].endswith("ai_agents.txt")`.

Because filtering happens **after** FAISS has picked the `fetch_k` nearest chunks, you get **fewer than `k` results** when fewer than `k` of those chunks match. Raise `fetch_k` if that happens. Chroma's `where` filter doesn't have this problem, because it filters inside the database.

## Strengths and limits

**Strengths**

- Very fast nearest-neighbour search, on CPU or GPU
- Scales from a few chunks to billions of vectors with IVF, HNSW and PQ indexes
- Lets you choose between exact and approximate search, and between speed and memory
- Runs in your Python process: one `pip install`, no server
- Open source (MIT license), widely used and maintained by Meta

**Limits to keep in mind**

- **A library, not a database.** No server, no user accounts, and no safe way for several processes to write at once.
- **Manual saving.** Nothing is written to disk until you call `save_local()`.
- **Filtering happens after the search.** LangChain filters metadata in Python, so filtered searches can return fewer than `k` results.
- **No upsert.** Updating a chunk means deleting it and adding it again. Some index types, such as HNSW, can't delete at all.
- **Everything lives in RAM.** The whole index is loaded into memory, so very large collections need a big machine or a compressed (PQ) index.
- **Pickle files.** The LangChain docstore is saved as a pickle, so only load saved stores you trust.

## Key takeaways

- **FAISS** is Meta's open-source library for fast vector search. It stores **only vectors**, and LangChain's `FAISS` class adds the **docstore** (text and metadata) and the id mapping.
- The default index is **`IndexFlatL2`**: exact brute-force search that returns **squared L2 distances** (lower = more similar).
- For bigger collections, approximate indexes (**IVF, HNSW, PQ**) trade a little accuracy for speed or memory. IVF and PQ must be **trained** first.
- For cosine scores, use `normalize_L2=True` with `DistanceStrategy.MAX_INNER_PRODUCT`. With normalised OpenAI embeddings, the default L2 index gives the same ranking.
- **Saving is manual**: `save_local()` writes `index.faiss` + `index.pkl`, and `load_local(..., allow_dangerous_deserialization=True)` loads them back.
- **No upsert**: check ids before calling `add_documents`, and delete a chunk before adding a new version of it.
- Metadata filters are applied **after** the search, on the `fetch_k` nearest chunks.
- `as_retriever()` plugs FAISS into the same LCEL and conversational RAG chains we built with Chroma.

**Next:** we'll build a FAISS vector store from our text chunks, save and load it, and run similarity searches.

In [1]:
## Setup: shared project paths from src.config, which also loads the API key from .env
from pathlib import Path

from src.config import PROJECT_ROOT, RAW_TEXT_DIR, FAISS_PATH

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp


In [2]:
## Load libraries
import warnings
warnings.filterwarnings("ignore")

# LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

## Data Ingestion

In [3]:
## Load the text files
from langchain_community.document_loaders import DirectoryLoader

doc_dir = RAW_TEXT_DIR
loader = DirectoryLoader(
    doc_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

documents = loader.load()
print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"  {doc.metadata['source']}: {len(doc.page_content)} characters")
print(f"\nFirst document preview:\n{documents[0].page_content[:200]}")

Loaded 4 documents
  data/raw/text/ai_agents.txt: 4117 characters
  data/raw/text/machine_learning_algorithms.txt: 4220 characters
  data/raw/text/reinforcement_learning.txt: 3909 characters
  data/raw/text/deep_learning.txt: 2505 characters

First document preview:
AI Agents: An Introduction

An AI agent is a system that uses a large language model (LLM) to decide what to do in order to reach a goal. A normal chatbot answers one message at a time. An agent can p


In [4]:
## Add a "topic" to each document's metadata (the chunks inherit it when we split)
topic_by_file = {
    "ai_agents.txt": "agents",
    "machine_learning_algorithms.txt": "ml",
    "reinforcement_learning.txt": "rl",
    "deep_learning.txt": "dl",
}

for doc in documents:
    doc.metadata["topic"] = topic_by_file[Path(doc.metadata["source"]).name]
    print(f"  {doc.metadata}")

  {'source': 'data/raw/text/ai_agents.txt', 'topic': 'agents'}
  {'source': 'data/raw/text/machine_learning_algorithms.txt', 'topic': 'ml'}
  {'source': 'data/raw/text/reinforcement_learning.txt', 'topic': 'rl'}
  {'source': 'data/raw/text/deep_learning.txt', 'topic': 'dl'}


In [5]:
## Split the documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],   # try paragraphs first, then lines, then words, then characters
)

chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")
print(f"Content : {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 44 chunks
Content : AI Agents: An Introduction

An AI agent is a system that uses a large language model (LLM) to decide what to do in order to reach a goal. A normal cha...
Metadata: {'source': 'data/raw/text/ai_agents.txt', 'topic': 'agents'}


In [6]:
## load embedding model

embedding = OpenAIEmbeddings(model="text-embedding-3-small")
embedding

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x12be99a90>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x12be9a7b0>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [7]:
## Embed the chunks with embed_documents
texts = [chunk.page_content for chunk in chunks]
vectors = embedding.embed_documents(texts)

print(f"Created {len(vectors)} embeddings")
print(f"Dimensions per vector: {len(vectors[0])}")
print(f"First 5 values of the first vector: {vectors[0][:5]}")

Created 44 embeddings
Dimensions per vector: 1536
First 5 values of the first vector: [-0.029083251953125, -0.0082244873046875, 0.0218963623046875, 0.00531005859375, 0.0103607177734375]


## Create FAISS Store

In [8]:
faiss_dir = FAISS_PATH

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
)
vectorstore.save_local(faiss_dir)

print(f"Saved {vectorstore.index.ntotal} vectors to {faiss_dir}")


Saved 44 vectors to vectorstore/faiss_index


In [9]:
vectorstore = FAISS.load_local(
    faiss_dir,
    embedding,                              # must be the same model that built the store
    allow_dangerous_deserialization=True,   # index.pkl is a pickle: only load files you created
)


In [10]:
query = "What is deep learning"

results = vectorstore.similarity_search(query, k=3)
print(results)

[Document(id='chunk_35', metadata={'source': 'data/raw/text/deep_learning.txt', 'topic': 'dl'}, page_content='Deep Learning: An Introduction\n\nDeep learning is a subfield of machine learning that uses artificial neural networks with many layers to learn patterns from data. The word "deep" refers to the number of layers stacked between the input and the output. Each layer learns a more abstract representation of the data than the one before it.'), Document(id='chunk_41', metadata={'source': 'data/raw/text/deep_learning.txt', 'topic': 'dl'}, page_content='Applications\nDeep learning is used in computer vision, natural language processing, speech recognition, machine translation, recommendation systems, medical image analysis, self-driving cars, and drug discovery.'), Document(id='chunk_43', metadata={'source': 'data/raw/text/deep_learning.txt', 'topic': 'dl'}, page_content='Conclusion\nDeep learning has transformed artificial intelligence by allowing machines to learn directly from raw 

In [11]:
## Similarity search: top 3 chunks
query = "What is reinforcement learning?"

results = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"--- Result {i} ---")
    print(f"Source : {doc.metadata.get('source', 'unknown')}")
    print(f"Content: {doc.page_content[:300]}...\n")


Query: What is reinforcement learning?

--- Result 1 ---
Source : data/raw/text/reinforcement_learning.txt
Content: Reinforcement Learning: An Introduction

Reinforcement learning (RL) is a branch of machine learning in which an agent learns to make decisions by interacting with an environment. Instead of learning from labeled examples, the agent tries actions, receives rewards or penalties, and gradually learns ...

--- Result 2 ---
Source : data/raw/text/machine_learning_algorithms.txt
Content: Reinforcement Learning
In reinforcement learning, an agent learns by interacting with an environment. It takes actions, receives rewards or penalties, and learns a strategy that maximizes total reward over time. Common algorithms include Q-Learning, Deep Q-Networks (DQN), and Proximal Policy Optimiz...

--- Result 3 ---
Source : data/raw/text/reinforcement_learning.txt
Content: Conclusion
Reinforcement learning teaches agents to make a sequence of good decisions through trial and error. Combin

In [12]:
## Similarity search with scores
results_with_score = vectorstore.similarity_search_with_score(query, k=3)

for i, (doc, score) in enumerate(results_with_score, start=1):
    print(f"--- Result {i} | distance: {score:.4f} ---")
    print(f"Source : {doc.metadata.get('source', 'unknown')}")
    print(f"Content: {doc.page_content[:300]}...\n")


--- Result 1 | distance: 0.6486 ---
Source : data/raw/text/reinforcement_learning.txt
Content: Reinforcement Learning: An Introduction

Reinforcement learning (RL) is a branch of machine learning in which an agent learns to make decisions by interacting with an environment. Instead of learning from labeled examples, the agent tries actions, receives rewards or penalties, and gradually learns ...

--- Result 2 | distance: 0.6863 ---
Source : data/raw/text/machine_learning_algorithms.txt
Content: Reinforcement Learning
In reinforcement learning, an agent learns by interacting with an environment. It takes actions, receives rewards or penalties, and learns a strategy that maximizes total reward over time. Common algorithms include Q-Learning, Deep Q-Networks (DQN), and Proximal Policy Optimiz...

--- Result 3 | distance: 0.7468 ---
Source : data/raw/text/reinforcement_learning.txt
Content: Conclusion
Reinforcement learning teaches agents to make a sequence of good decisions through trial 

In [13]:
## Metadata filtering: only search chunks with a given topic
topic_filters = {
    "topic = ml":       {"topic": "ml"},
    "topic = ml or dl": {"topic": {"$in": ["ml", "dl"]}},
}

for label, topic_filter in topic_filters.items():
    filtered_results = vectorstore.similarity_search_with_score(
        query,
        k=3,
        filter=topic_filter,
        fetch_k=20,   # FAISS fetches the 20 nearest chunks, then LangChain keeps the ones that match the filter
    )

    print(f"===== Filter: {label} =====\n")
    for i, (doc, score) in enumerate(filtered_results, start=1):
        print(f"--- Result {i} | distance: {score:.4f} ---")
        print(f"Topic  : {doc.metadata.get('topic', 'unknown')}")
        print(f"Content: {doc.page_content[:300]}...\n")

===== Filter: topic = ml =====

--- Result 1 | distance: 0.6863 ---
Topic  : ml
Content: Reinforcement Learning
In reinforcement learning, an agent learns by interacting with an environment. It takes actions, receives rewards or penalties, and learns a strategy that maximizes total reward over time. Common algorithms include Q-Learning, Deep Q-Networks (DQN), and Proximal Policy Optimiz...

--- Result 2 | distance: 1.3022 ---
Topic  : ml
Content: Supervised Learning
In supervised learning, the model is trained on labeled data, meaning each input comes with the correct output. The goal is to predict the output for new, unseen inputs. Supervised tasks are either regression (predicting a number) or classification (predicting a category).

Linea...

--- Result 3 | distance: 1.3406 ---
Topic  : ml
Content: Conclusion
Machine learning algorithms are the foundation of modern AI systems. Understanding their strengths and weaknesses helps you choose the right tool for each problem, and more adv

## Build the RAG Chain

In [14]:
## Retriever, LLM and prompt
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatOpenAI(model="gpt-5", temperature=0)

custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question.
If you don't know the answer based on the context, say you don't know.
Mention the source file(s) you used.

Context:
{context}

Question: {question}

Answer:""")

In [15]:
## Format the retrieved chunks for the prompt, with a [Source: ...] label on each
def format_docs(docs):
    return "\n\n".join(f"[Source: {doc.metadata['source']}]\n{doc.page_content}" for doc in docs)

In [16]:
## Build the chain using LCEL
rag_chain_lcel = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough()}
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x12bd9bb10>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question.\nIf you don't know the answer based on the context, say you don't know.\nMention the source file(s) you used.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.1', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-5', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 27200

In [17]:
## Ask a question
question = "What is reinforcement learning?"
answer = rag_chain_lcel.invoke(question)

print(f"Question: {question}\n")
print(f"Answer:\n{answer}")

Question: What is reinforcement learning?

Answer:
Reinforcement learning is a branch of machine learning where an agent learns by interacting with an environment—taking actions, receiving rewards or penalties—and learns a strategy that maximizes total reward over time.

Sources: data/raw/text/reinforcement_learning.txt; data/raw/text/machine_learning_algorithms.txt


In [18]:
## RAG chain with a metadata filter: the retriever only returns chunks with topic = dl
dl_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3, "filter": {"topic": "dl"}, "fetch_k": 20}
)

dl_rag_chain = (
    {"context": dl_retriever | format_docs,
     "question": RunnablePassthrough()}
    | custom_prompt
    | llm
    | StrOutputParser()
)

# deep_learning.txt doesn't cover reinforcement learning, so the LLM should say it doesn't know
for question in ["What are the common deep learning architectures?", "What is reinforcement learning?"]:
    print(f"Question: {question}\n")
    print(f"Answer:\n{dl_rag_chain.invoke(question)}\n")

Question: What are the common deep learning architectures?

Answer:
- Convolutional Neural Networks (CNNs) for images (e.g., image classification, object detection)
- Recurrent Neural Networks (RNNs) and Long Short-Term Memory networks (LSTMs) for sequential data (e.g., text, speech, time series)

Source: data/raw/text/deep_learning.txt

Question: What is reinforcement learning?

Answer:
I don’t know based on the provided context. The excerpts only discuss deep learning and RAG, not reinforcement learning.

Sources used:
- data/raw/text/deep_learning.txt



In [19]:
## LCEL chain that returns the answer AND the retrieved chunks (with their metadata)
answer_chain = (
    RunnablePassthrough.assign(context=lambda x: format_docs(x["docs"]))   # Documents -> context string
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_with_sources = (
    {"docs": retriever, "question": RunnablePassthrough()}   # {"docs": [Document, ...], "question": "..."}
    | RunnablePassthrough.assign(answer=answer_chain)         # adds "answer", keeps "docs" and "question"
)

In [20]:
## Helper: ask the RAG chain a question (async) and show the answer with each chunk's metadata
async def query_rag_with_sources(question, chain=rag_chain_with_sources):
    """Run the chain with ainvoke, print the answer and the metadata of the chunks it used."""
    result = await chain.ainvoke(question)

    print(f"Question: {result['question']}\n")
    print(f"Answer:\n{result['answer']}")

    print(f"\nRetrieved {len(result['docs'])} chunks:")
    for i, doc in enumerate(result["docs"], start=1):
        print(f"\n --- Chunk {i} --- id: {doc.id}")
        for key, value in doc.metadata.items():
            print(f"   {key:<8}: {value}")
        print(f"   content : {doc.page_content[:200]}...")

    return result


result = await query_rag_with_sources("What is reinforcement learning?")

Question: What is reinforcement learning?

Answer:
Reinforcement learning is a branch of machine learning where an agent learns through trial and error by interacting with an environment: it takes actions, receives rewards or penalties, and learns a strategy (policy) that maximizes total reward over time.

Sources: data/raw/text/reinforcement_learning.txt; data/raw/text/machine_learning_algorithms.txt

Retrieved 3 chunks:

 --- Chunk 1 --- id: chunk_23
   source  : data/raw/text/reinforcement_learning.txt
   topic   : rl
   content : Reinforcement Learning: An Introduction

Reinforcement learning (RL) is a branch of machine learning in which an agent learns to make decisions by interacting with an environment. Instead of learning ...

 --- Chunk 2 --- id: chunk_19
   source  : data/raw/text/machine_learning_algorithms.txt
   topic   : ml
   content : Reinforcement Learning
In reinforcement learning, an agent learns by interacting with an environment. It takes actions, receives rewards 

## Conversational RAG with chat history (LCEL)

The chains above forget each question as soon as they answer it, so a follow-up like *"What are its main challenges?"* fails. Here we build the same two steps as `create_history_aware_retriever` + `create_retrieval_chain` from the conversational memory notebook, but with LCEL:

```
{"question": "What are its main challenges?", "chat_history": [...]}
        │
        ▼
 1. condense_question    no history  → use the question as it is
        │                with history → LLM rewrites it: "What are the main challenges of reinforcement learning?"
        ▼
 2. retriever            FAISS searches with the standalone question → docs
        │
        ▼
 3. answer               qa_prompt (context + chat_history + question) | llm | StrOutputParser()
        │
        ▼
{"question", "chat_history", "standalone_question", "docs", "answer"}
```

In [21]:
## Step 1: rewrite a follow-up question into a standalone question before searching
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables import RunnableBranch

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the chat history and the latest user question, which might refer to the "
     "chat history, rewrite it as a standalone question that can be understood without "
     "the chat history. Do NOT answer the question: only rewrite it if needed, "
     "otherwise return it as is."),
    MessagesPlaceholder("chat_history"),   # the past HumanMessage / AIMessage objects go here
    ("human", "{question}"),
])

condense_question = RunnableBranch(
    (lambda x: not x["chat_history"], lambda x: x["question"]),   # no history: search with the question as it is
    contextualize_q_prompt | llm | StrOutputParser(),              # with history: let the LLM rewrite it first
)

In [22]:
## Step 2 and 3: search FAISS with the standalone question, then answer with the context AND the chat history
qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. Answer using only the retrieved context below. "
     "If the answer is not in the context, say you don't know based on the provided documents. "
     "Cite sources using the [Source: ...] labels. Keep the answer concise.\n\n"
     "Context:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

conversational_answer_chain = (
    RunnablePassthrough.assign(context=lambda x: format_docs(x["docs"]))   # Documents -> context string
    | qa_prompt
    | llm
    | StrOutputParser()
)

conversational_rag_chain = (
    RunnablePassthrough.assign(standalone_question=condense_question)
    | RunnablePassthrough.assign(docs=(lambda x: x["standalone_question"]) | retriever)
    | RunnablePassthrough.assign(answer=conversational_answer_chain)
)

In [23]:
## Helper: chat with memory (async), save each turn, and show what FAISS searched for
chat_history = []

async def chat_with_memory(question):
    """Answer with the chat history, save the turn, and print the chunks it used with their metadata."""
    result = await conversational_rag_chain.ainvoke({"question": question, "chat_history": chat_history})
    chat_history.extend([HumanMessage(content=question), AIMessage(content=result["answer"])])

    print(f"Question     : {question}")
    print(f"Searched with: {result['standalone_question']}\n")
    print(f"Answer:\n{result['answer']}\n")
    print("Retrieved chunks:")
    for doc in result["docs"]:
        print(f"  - {doc.id} | topic: {doc.metadata.get('topic')} | {doc.metadata.get('source')}")
    print()
    return result


result = await chat_with_memory("What is reinforcement learning?")
result = await chat_with_memory("What are its main challenges?")   # "its" = reinforcement learning

Question     : What is reinforcement learning?
Searched with: What is reinforcement learning?

Answer:
Reinforcement learning is a branch of machine learning where an agent learns through trial and error by interacting with an environment, taking actions, receiving rewards or penalties, and optimizing for long-term total reward. Common algorithms include Q-Learning, DQN, and PPO, with applications in game playing, robotics, and aligning large language models via RLHF. [Source: data/raw/text/reinforcement_learning.txt] [Source: data/raw/text/machine_learning_algorithms.txt]

Retrieved chunks:
  - chunk_23 | topic: rl | data/raw/text/reinforcement_learning.txt
  - chunk_19 | topic: ml | data/raw/text/machine_learning_algorithms.txt
  - chunk_34 | topic: rl | data/raw/text/reinforcement_learning.txt

Question     : What are its main challenges?
Searched with: What are the main challenges of reinforcement learning?

Answer:
- Sample inefficiency: needs many interactions, which can be slow/